In [2]:

# ===== STEP 1: Install required packages =====
!pip install -q transformers torch

# ===== STEP 2: Mount Google Drive (if files are in Drive) =====
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ===== STEP 3: Main Script =====
import os
import shutil
import torch
from pathlib import Path
from transformers import pipeline
from tqdm import tqdm
from functools import lru_cache

# Check GPU availability
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")
print(f"GPU Name: {torch.cuda.get_device_name(0) if device == 0 else 'N/A'}\n")

# Initialize model with GPU
print("Loading AI model on GPU...")
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device,
    batch_size=8  # Process multiple files at once
)
print("Model loaded!\n")

@lru_cache(maxsize=1000)
def fast_keyword_check(text_lower):
    property_keywords = {
        'property','land','immovable','plot','ownership','title','deed',
        'registry','possession','khasra','mutation','patwari','jamabandi',
        'encroachment','partition','inheritance','pre-emption','benami',
        'injunction','transfer of property act','land revenue act'
    }
    return len(property_keywords & set(text_lower.split())) >= 3

def strict_negative_filter(text):
    negative_terms = [
        "Article 186",
        "Presidential Reference",
        "criminal trial",
        "conviction",
        "murder trial",
        "constitutional jurisdiction",
        "bench",
        "judges",
        "amici curiae"
    ]
    return any(term.lower() in text.lower() for term in negative_terms)


def is_property_related_batch(texts, threshold=0.5):
    property_labels = [
        "property dispute", "land ownership", "title dispute",
        "inheritance of property", "sale of property",
        "land record mutation", "boundary dispute",
        "tenant landlord dispute", "real estate dispute"
    ]

    non_property_labels = [
        "criminal law", "service matters", "constitutional petitions",
        "tax matters", "family disputes"
    ]

    labels = property_labels + non_property_labels

    # Prepare text samples
    clean_texts = []
    for text in texts:
        if not text or len(text.strip()) < 100:
            clean_texts.append("")
        else:
            clean_texts.append(text[:2000])

    # Model inference
    try:
        outputs = classifier(clean_texts, labels, multi_label=True)
    except:
        outputs = [None] * len(clean_texts)

    results = []

    for idx, text in enumerate(texts):
        if not text or len(text.strip()) < 50:
            results.append(fast_keyword_check(text.lower() if text else ""))
            continue

        # ✅ Negative filter BEFORE ML classification logic
        sample = clean_texts[idx]
        if strict_negative_filter(sample):
            results.append(False)
            continue

        result = outputs[idx]
        if result is None:
            results.append(fast_keyword_check(text.lower()))
            continue

        # ✅ Check property confidence
        is_prop = False
        for lbl, score in zip(result["labels"], result["scores"]):
            if lbl in property_labels and score > threshold:
                is_prop = True
                break

        # ✅ Final decision using BOTH ML and keywords
        if is_prop and fast_keyword_check(text.lower()):
            results.append(True)
        else:
            results.append(False)

    return results

def keyword_fallback(text):
    """Fast keyword-based detection"""
    if not text:
        return False

    text_lower = text.lower()

    property_keywords = [
        'property', 'land', 'immovable', 'plot', 'estate',
        'ownership', 'title', 'deed', 'sale deed', 'gift deed',
        'transfer deed', 'conveyance', 'registry',
        'possession', 'occupancy', 'adverse possession',
        'khasra', 'khewat', 'khatauni',
        'mutation', 'patwari', 'revenue', 'jamabandi',
        'fard', 'intiqal', 'tehsildar',
        'agricultural land', 'residential property', 'commercial property',
        'house', 'building', 'apartment', 'flat',
        'encroachment', 'trespass', 'boundary dispute', 'partition',
        'easement', 'right of way', 'inheritance', 'succession',
        'pre-emption', 'shufaa', 'benami', 'specific performance',
        'injunction', 'suit for possession', 'suit for declaration',
        'transfer of property act', 'land revenue act',
        'land acquisition act', 'colonization act',
        'succession act', 'pre-emption act'
    ]

    matches = sum(1 for kw in property_keywords if kw in text_lower)
    return matches >= 3


def filter_property_judgments_gpu(source_folder, output_folder, batch_size=16):
    """
    GPU-accelerated filtering with batch processing

    Args:
        source_folder: Path to judgment files
        output_folder: Output path
        batch_size: Number of files to process at once
    """

    # Create output folder
    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    # Get all txt files
    source_path = Path(source_folder)
    txt_files = list(source_path.glob('*.txt'))
    total_files = len(txt_files)

    if total_files == 0:
        print(f"❌ No .txt files found in {source_folder}")
        return

    print(f"Found {total_files} judgment files")
    print(f"Processing in batches of {batch_size}...\n")

    property_count = 0

    # Process in batches
    for i in tqdm(range(0, total_files, batch_size), desc="Processing batches"):
        batch_files = txt_files[i:i+batch_size]
        batch_texts = []
        valid_files = []

        # Read batch
        for file_path in batch_files:
            try:
                with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                    content = f.read()
                batch_texts.append(content)
                valid_files.append(file_path)
            except:
                continue

        # Classify batch
        if batch_texts:
            results = is_property_related_batch(batch_texts)

            # Copy property-related files
            for file_path, is_property in zip(valid_files, results):
                if is_property:
                    destination = output_path / file_path.name
                    shutil.copy2(file_path, destination)
                    property_count += 1

    # Summary
    print("\n" + "="*60)
    print("✓ DONE!")
    print(f"✓ Total files processed: {total_files}")
    print(f"✓ Property-related files found: {property_count}")
    print(f"✓ Percentage: {(property_count/total_files)*100:.1f}%")
    print(f"✓ Files saved to: {output_path}")
    print("="*60)

    return property_count


# ===== STEP 4: Configure and Run =====

# CHANGE THESE PATHS TO YOUR ACTUAL PATHS
SOURCE_FOLDER = "/content/drive/MyDrive/LegalMateData/scj"  # Your judgment files location
OUTPUT_FOLDER = "/content/drive/MyDrive/LegalMateData/scj-property"  # Where to save filtered files

# Run the filter
property_count = filter_property_judgments_gpu(
    SOURCE_FOLDER,
    OUTPUT_FOLDER,
    batch_size=16  # Increase if you have more GPU memory
)

# Optional: Download results
# from google.colab import files
# !zip -r property_judgments.zip "{OUTPUT_FOLDER}"
# files.download('property_judgments.zip')

Using device: GPU
GPU Name: Tesla T4

Loading AI model on GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Model loaded!

Found 2798 judgment files
Processing in batches of 16...



Processing batches: 100%|██████████| 175/175 [1:03:56<00:00, 21.93s/it]


✓ DONE!
✓ Total files processed: 2798
✓ Property-related files found: 254
✓ Percentage: 9.1%
✓ Files saved to: /content/drive/MyDrive/LegalMateData/scj-property
